# Insurance Claim Evaluation Agent

**1. Problem Statement**
Insurance claim processing is traditionally manual — adjusters retrieve policy documents, verify treatment eligibility, validate costs against coverage limits, apply deductible rules, and produce a compliant decision. This is slow, inconsistent, and doesn't scale with claim volume.

**2. What the Agent Does**
This agent automates first-level insurance claim evaluation. Given a claim, it autonomously retrieves policy details, checks treatment eligibility, calculates the payable amount, and delivers a final APPROVED or REJECTED verdict — within a single invocation.

**3. Tool invocation**
- check_policy_coverage --> Checks if a treatment is covered based on policy rules.

## Setup (Installation, import)

In [1]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)
# !pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24

1.2.10
0.4.1


In [2]:
import os
import json
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')

/Users/gourasundarmohanty/miniconda3/envs/py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_core.prompts import ChatPromptTemplate
# from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import tool

In [4]:
# The LLM (Agent Brain)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Define tools

In [5]:

# Domain Tool Using @tool
@tool
def check_policy_coverage(details: str) -> str:
    """
    Checks if a treatment is covered based on policy rules.
    Expected format: "treatment=<name>, cost=<amount>, policy_limit=<amount>"
    """
    try:
        data = dict(item.split("=") for item in details.replace(" ", "").split(","))
        cost = float(data["cost"])
        limit = float(data["policy_limit"])

        if cost <= limit:
            return f"Approved: Cost {cost} is within policy limit {limit}."
        else:
            return f"Rejected: Cost {cost} exceeds policy limit {limit}."
    
    except Exception as e:
        return f"Invalid input format. Error: {str(e)}"

tools = [check_policy_coverage]

## Define Agent prompt

In [7]:
# Agent Prompt Template
prompt = ChatPromptTemplate.from_template("""
You are a Healthcare Insurance Claim Decision Agent.

Follow this reasoning process:
1. Understand the request.
2. Decide whether the policy-checking tool is needed.
3. If yes, call the tool.
4. Interpret the tool’s response.
5. Provide a final human-friendly approval or rejection outcome.

Make the decision clear, justified, and formatted for compliance officers.

User Request: {input},
("placeholder", "{agent_scratchpad}"
""")

## Create Agents

* `create_tool_calling_agent()` **creates the agent**, defining how it should think and which tools it can use.
* `AgentExecutor()` **runs the agent**, allowing it to execute reasoning and tool calls with controls like iterations and logging.

Agent = brain

AgentExecutor = brain + execution engine

In [8]:
!pip install langchain-classic

In [11]:
# Create Agent
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
# from langchain_core.agents import AgentExecutor, create_tool_calling_agent

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)


# Agent Executor Runtime
# 'verbose' shows step logs, and 'max_iterations' limits how many reasoning steps it can take.
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=1)

# Run Example
response = executor.invoke({
    "input": "Evaluate claim: treatment=Physical Therapy, cost=1200, policy_limit=2000"
})

print("\nFINAL OUTPUT:", response["output"])



> Entering new AgentExecutor chain...

Invoking: `check_policy_coverage` with `{'details': 'treatment=Physical Therapy, cost=1200, policy_limit=2000'}`


Approved: Cost 1200.0 is within policy limit 2000.0.

> Finished chain.

FINAL OUTPUT: Agent stopped due to max iterations.


In [ ]:
# # some more test cases for
# input1: "Evaluate claim: treatment=Physical Therapy, cost=1200, policy_limit=2000"
# input 2: "Evaluate claim: treatment=Chemotherapy, cost=5000, policy_limit=4000"

The agent doesn't just answer.
It follows a cognitive workflow:
Interpret the query
Reason about whether a tool is needed
Call the tool (if required)
Observe the result
Reflect
Produce the final answer

Each cycle (Thought → Action → Observation → Reflection) counts as an iteration.
LangChain sets a default max_iterations limit (usually 15 or 20 depending on the version) to prevent: use max_iterations=1 in AgentExecutor


# Agent: Healthcare Claim Decision AI (Shows multistep resoning)
 - This agent first retrieves policy details using the **get_policy_details tool**, checks if the requested treatment is eligible under the policy, 
 - calculates the final claim amount using the **calculate_claim** tool, and finally provides an approval or rejection decision based on the policy rules and calculated amount.

In [16]:
#mock db database
POLICY_DB = {

    # 1. Normal adult policy — simple eligibility
    "P123": {
        "coverage": 0.85,
        "eligible_treatments": ["MRI", "Surgery", "Blood Test"],
        "deductible": 2000,
        "waiting_period_conditions": [],
        "pre_existing_conditions": [],
        "age_limit": None
    },

    # 2. Budget policy — limited procedures
    "P876": {
        "coverage": 0.60,
        "eligible_treatments": ["X-Ray", "Blood Test"],
        "deductible": 5000,
        "waiting_period_conditions": ["Surgery"],
        "pre_existing_conditions": ["Diabetes"],
        "age_limit": None
    },

    # 3. Senior citizen plan — high coverage but restrictions
    "SENIOR901": {
        "coverage": 0.90,
        "eligible_treatments": ["MRI", "Cardiac Surgery", "CT Scan", "Blood Test"],
        "deductible": 8000,
        "waiting_period_conditions": ["Joint Replacement"],
        "pre_existing_conditions": [],
        "age_limit": 75
    },

    # 4. Corporate policy — most generous coverage, no deductible
    "CORP555": {
        "coverage": 1.00,
        "eligible_treatments": ["MRI", "Surgery", "Dental Surgery", "Therapy", "ICU", "Blood Test"],
        "deductible": 0,
        "waiting_period_conditions": [],
        "pre_existing_conditions": [],
        "age_limit": None
    },

    # 5. Maternity policy — special category
    "MAT2024": {
        "coverage": 0.70,
        "eligible_treatments": ["Delivery (Normal)", "C-Section", "Ultrasound"],
        "deductible": 10000,
        "waiting_period_conditions": ["C-Section"],
        "pre_existing_conditions": ["Hypertension"],
        "age_limit": 45
    },

    # 6. Accident-Only Coverage — restricts illness-based claims
    "ACC100": {
        "coverage": 0.95,
        "eligible_treatments": ["Fracture Treatment", "Emergency Surgery", "ICU"],
        "deductible": 3000,
        "waiting_period_conditions": [],
        "pre_existing_conditions": [],
        "age_limit": None,
        "coverage_type": "Accident Only"
    },

    # 7. Chronic Care Plan — approvals require conditions match the policy
    "CHRONIC777": {
        "coverage": 0.65,
        "eligible_treatments": ["Dialysis", "Cardiac Monitoring", "Blood Test"],
        "deductible": 1500,
        "waiting_period_conditions": [],
        "pre_existing_conditions": ["Kidney Failure"],
        "age_limit": None
    }
}


In [17]:
@tool
def get_policy_details(policy_id: str) -> str:
    """Retrieve insurance policy coverage %, eligible treatments."""
    return json.dumps(POLICY_DB.get(policy_id, "Policy not found"))


@tool
def calculate_claim(amount: str) -> str:
    """Calculate final claim amount based on % coverage (format: 'coverage amount')."""
    try:
        coverage, bill = map(float, amount.split())
        return str(round(coverage * bill, 2))
    except:
        return "Invalid format. Use: <coverage> <bill_amount>"

In [18]:
claim_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a Healthcare Claim Approval Agent.
Your job:
1. Retrieve policy details
2. Check treatment eligibility
3. Calculate final payable amount
4. Give final decision: APPROVED or REJECTED
Think step-by-step and use tools ONLY if necessary.
"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])


In [27]:
tools = [get_policy_details, calculate_claim]

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=claim_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=1
)

In [28]:
query = "A patient with policy P123 requested MRI worth 50,000 INR. Should the claim be approved?"
result = agent_executor.invoke({"input": query})

print("\nFINAL OUTPUT:")
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `get_policy_details` with `{'policy_id': 'P123'}`


{"coverage": 0.85, "eligible_treatments": ["MRI", "Surgery", "Blood Test"], "deductible": 2000, "waiting_period_conditions": [], "pre_existing_conditions": [], "age_limit": null}

> Finished chain.

FINAL OUTPUT:
Agent stopped due to max iterations.


In [ ]:
# # Sample test queries for the Claim Agent

# test_queries = [
#     "A patient with policy P876 requested Surgery worth 120,000 INR. Should the claim be approved?",
#     "A 78-year-old patient with policy SENIOR901 requested a CT Scan costing 35,000 INR. Should this claim be processed?",
#     "A policyholder under CORP555 submitted a claim for Dental Surgery worth 85,000 INR. Is this eligible for approval?",
#     "A patient with policy ACC100 filed a claim for a Blood Test costing 2,500 INR due to fever. Should this request be accepted?",
#     "A policyholder with MAT2024 and a pre-existing condition of Hypertension requested a claim for a C-Section delivery worth 95,000 INR. Should the claim be approved?"
# ]

# test_queries
# for query in test_queries:
#     print(f"\nProcessing Query: {query}")
#     result = agent_executor.invoke({"input": query})
#     print("FINAL OUTPUT:", result["output"])